# 02. MEG Emotion Recognition


## What This Notebook Does

This notebook keeps the MEG pipeline as simple as possible:

1. load the MEG CSV file
2. create a synthetic CSV if a real file is missing
3. split the data into `X` and `y`
4. scale the features
5. train one SVM model
6. save the trained files


## Step 1: Setup


In [ ]:
from pathlib import Path
import sys

current_dir = Path.cwd().resolve()
possible_dirs = [current_dir, current_dir / "NeuroSense" / "notebooks"]
notebooks_dir = next((path for path in possible_dirs if path.exists() and path.name == "notebooks"), None)
if notebooks_dir is None:
    raise FileNotFoundError("Start Jupyter from the project root or from NeuroSense/notebooks.")

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook()
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
RANDOM_STATE = ctx["random_state"]

print("Datasets directory:", DATASETS_DIR)
print("Artifacts directory:", ARTIFACTS_DIR)


## Step 2: Import The Libraries


In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC


## Step 3: Load Or Create The Dataset

If a MEG CSV is not available, we create a small synthetic one so the ML pipeline can still be demonstrated.


In [ ]:
def make_synthetic_meg_dataframe(random_state):
    rng = np.random.default_rng(random_state)

    labels = rng.choice(["POSITIVE", "NEGATIVE", "NEUTRAL"], size=600, p=[0.34, 0.33, 0.33])
    features = rng.normal(0, 1, size=(600, 50))

    features[labels == "POSITIVE", :10] += 0.5
    features[labels == "NEGATIVE", 10:20] += 0.5
    features[labels == "NEUTRAL", 20:30] += 0.2
    features += rng.normal(0, 0.3, size=features.shape)

    df = pd.DataFrame(features, columns=[f"meg_f{i}" for i in range(50)])
    df["label"] = labels
    return df


meg_path = DATASETS_DIR / "meg" / "meg_features.csv"
meg_path.parent.mkdir(parents=True, exist_ok=True)
using_synthetic_data = False

if not meg_path.exists():
    using_synthetic_data = True
    df = make_synthetic_meg_dataframe(RANDOM_STATE)
    df.to_csv(meg_path, index=False)
    print("No real MEG CSV was found, so a synthetic demo dataset was created.")
else:
    df = pd.read_csv(meg_path)
    feature_columns = [column for column in df.columns if column != "label"]
    using_synthetic_data = all(column.startswith("meg_f") for column in feature_columns)

print("Dataset shape:", df.shape)
print("Using synthetic data:", using_synthetic_data)
df.head()


## Step 4: Prepare `X` And `y`


In [ ]:
if "label" not in df.columns:
    raise ValueError("The MEG CSV must contain a 'label' column.")

X = df.drop(columns=["label"]).fillna(0.0)
y = df["label"].astype(str)

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=RANDOM_STATE,
)

print("Feature matrix shape:", X.shape)
print("Class names:", list(encoder.classes_))


## Step 5: Scale The Features


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Step 6: Train The Model


In [ ]:
model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)


## Step 7: Check The Result


In [ ]:
train_accuracy = accuracy_score(y_train, model.predict(X_train_scaled))
test_accuracy = accuracy_score(y_test, y_pred)

print("MEG train accuracy:", round(train_accuracy, 4))
print("MEG test accuracy:", round(test_accuracy, 4))
if using_synthetic_data:
    print("Note: this accuracy comes from synthetic demo data, not from a real MEG acquisition.")

print(classification_report(y_test, y_pred, target_names=encoder.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=encoder.classes_,
    cmap="Purples",
    xticks_rotation=20,
)
plt.title("MEG confusion matrix")
plt.tight_layout()
plt.show()


## Step 8: Save The Trained Files


In [ ]:
artifact_dir = ARTIFACTS_DIR / "meg"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "meg_model.pkl")
joblib.dump(scaler, artifact_dir / "meg_scaler.pkl")
joblib.dump(encoder, artifact_dir / "meg_label_encoder.pkl")

metadata = {
    "data_source": "synthetic" if using_synthetic_data else "real",
    "data_source_note": (
        "Synthetic MEG-like data was used because no real MEG CSV was found."
        if using_synthetic_data
        else "The model was trained on the MEG CSV found in the dataset folder."
    ),
    "evaluation_method": "80/20 Stratified Random Split",
    "train_accuracy": round(float(train_accuracy), 4),
    "test_accuracy": round(float(test_accuracy), 4),
}
with open(artifact_dir / "meg_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Saved MEG files to:", artifact_dir)
